# Streaming a response

Goal: show the model's words as they arrive instead of waiting for the finished reply. Same
`responses` API as the last notebook, with `stream=True` and a loop over events.

## Load the API key

`.env` sits at the repo root; `find_dotenv` walks up the folders to find it. The key is never
printed - this repo is public.

In [ ]:
from dotenv import load_dotenv, find_dotenv

# walk up from this notebook's folder until a .env is found
env_path = find_dotenv(usecwd=True)
assert env_path, ".env not found in this folder or any parent"
load_dotenv(env_path)

## Create the client

`OpenAI()` picks `OPENAI_API_KEY` up from the environment, so the constructor takes no arguments.

In [ ]:
from openai import OpenAI

openai_client = OpenAI()

## Stream plain text

`stream=True` turns the call into an iterator of events. Two kinds matter here: the ones carrying
a `delta`, which is the next piece of text, and the final one carrying the finished `response`.
`hasattr` is the filter because the stream also sends events this loop has no use for.

In [ ]:
messages = [
    {"role": "user", "content": "tell me a bad time story about a unicorn"}
]

stream = openai_client.responses.create(
    model='gpt-4o-mini',
    input=messages,
    stream=True
)

response = None
for event in stream:
    if hasattr(event, 'delta'):
        print(event.delta, end='')
    if hasattr(event, 'response'):
        response = event.response

The text lands piece by piece instead of all at once, which is the whole point of the flag.
`response` stays `None` until the final event, so anything needing the complete reply has to wait
for the loop to end. The story differs on every run.

# Structured output, streamed

Goal: ask for a schema and still get a stream. The model now has to produce JSON, which changes
what the deltas look like.

## Describe the shape you want

Two Pydantic models: an article made of sections. `Field(description=...)` travels to the model as
part of the schema, so it reads as an instruction about what belongs in that field.

In [ ]:
from pydantic import BaseModel, Field

In [ ]:
class ArticleSection(BaseModel):
    header: str
    text: str = Field(description="text of the section in markdown")

class ArticleResponse(BaseModel):
    title: str
    subtitle: str
    sections: list[ArticleSection]

## Stream with a schema

`responses.stream(...)` with `text_format`, rather than `create(..., stream=True)`. The context
manager is what lets the SDK hand back a parsed object once the last event has landed.

In [ ]:
messages = [
    {"role": "user", "content": "tell me a bad time story about a unicorn"}
]

with openai_client.responses.stream(
    model="gpt-4o-mini",
    input=messages,
    text_format=ArticleResponse,
) as stream:
    response = None
    for event in stream:
        if hasattr(event, 'delta'):
            print(event.delta, end='')
        if hasattr(event, 'response'):
            response = event.response

The deltas are raw JSON now, printed exactly as they arrive - keys, braces and escaped quotes.
It streams, but there is nothing here a reader would want to look at. That is the problem the rest
of the notebook solves.

## Read the parsed object

`output_parsed` is the `ArticleResponse` instance the SDK built from the completed JSON - so this
is attribute access, not parsing.

In [ ]:
story = response.output_parsed

## Print it as markdown

Walk the sections and add the markdown prefixes by hand. Readable at last, but only after the
whole response arrived - the same wait `stream=True` was supposed to remove.

In [ ]:
print('# ' + story.title)
print(story.subtitle)
print()

for section in story.sections:
    print('## ' + section.header)
    print()
    print(section.text)
    print()
    print()

# Parse the JSON as it arrives

Goal: turn those raw JSON deltas into readable output mid-stream. Deltas don't split on field
boundaries, so this needs a parser that accepts half a JSON document and reports what it has
so far.

## Install the parser

`jaxn` is an incremental JSON parser - feed it characters and it calls you back as fields open
and close.

In [ ]:
!uv add jaxn

`uv add` on its own was enough for the import below to work, so this kernel is the project
`.venv`. That is different from `02-rag`, where every `uv add` needed a `%pip install` after it to
reach the kernel's interpreter.

In [ ]:
from jaxn import JSONParserHandler, StreamingJSONParser

## A finished response to practise on

One complete JSON response, pasted in as a string. Feeding a known document by hand makes the
parser's behaviour easy to watch before a live stream is behind it.

In [ ]:
raw_json = """
{"title":"The Misadventures of Sparkle, the Unicorn","subtitle":"A Tale of Trouble and Lessons","sections":[{"header":"Once Upon a Time","text":"In a magical land far, far away, there lived a young unicorn named Sparkle. She was known for her shimmering coat and dazzling horn, but she had one major flaw: a mischievous streak that often led to chaos."},{"header":"The Forbidden Forest","text":"One sunny morning, Sparkle overheard her friends talking about the Forbidden Forest, a place where no unicorn dared to go. They said it was filled with danger and dark magic. But Sparkle, intrigued by the thrill, decided to venture into the forest alone."},{"header":"A Wretched Encounter","text":"As she galloped deeper into the forest, Sparkle found herself surrounded by eerie shadows and ominous sounds. Suddenly, she stumbled upon a group of grumpy trolls who were not too happy to see an intruder. Instead of being scared, Sparkle decided to play a prank on them."},{"header":"The Troll Trap","text":"Using her magic, she conjured up illusions of beautiful flowers and sparkling jewels. The trolls, greedy and foolish, fell for her trick and rushed toward the illusions. However, in doing so, they knocked over a giant boulder, blocking Sparkle's way back."},{"header":"The Consequences","text":"Trapped and surrounded by trolls who were now angrier than ever, Sparkle realized her mischief had backfired. Her only hope was to escape the trolls and unblock the path, but her magic was drained from the prank."},{"header":"A Lesson Learned","text":"Just when things seemed bleak, Sparkle remembered her true friends who would come looking for her. She calmed herself, using her wit instead of magic. She negotiated with the trolls to let her go, promising never to trespass again."},{"header":"Home Safe, but Changed","text":"With the trolls distracted by a fine tale of treasures far beyond the forest, Sparkle dashed away and eventually found her way home. Though she escaped, she learned that her mischievous nature could lead her into great trouble. From that day on, Sparkle decided to be a better friend and use her magic wisely."},{"header":"The End","text":"And so, the tale of Sparkle the unicorn spread throughout the land, reminding everyone that sometimes, the thrill of adventure can come with consequences. And being true to oneself is the most magical lesson of all."}]}
""".strip()

## Decide what to do with each piece

The handler is a set of callbacks on a parse in progress. `on_field_end` fires when a field is
complete, which is the right moment to print the title and each section header with its markdown
prefix. `on_value_chunk` fires on every fragment of a value, and that is what makes the section
text appear as it arrives. `path` says where in the document you are - `''` at the top level,
`/sections` inside the array - and is how the same field name in two places stays distinguishable.

In [ ]:
from typing import Any, Dict

class ArticleResponseHandler(JSONParserHandler):

    def on_field_end(self, path: str, field_name: str, value: str, parsed_value: Any = None) -> None:
        if path == '':
            if field_name == 'title':
                print(f'# {value}')
            if field_name == 'subtitle':
                print(value)
        if path == '/sections' and field_name == 'header':
            print(f'\n\n## {value}\n')

    def on_value_chunk(self, path: str, field_name: str, chunk: str) -> None:
        if path == '/sections' and field_name == 'text':
            print(chunk, end='', flush=True)

## Wire the handler to a parser

`StreamingJSONParser` holds the parse state and calls the handler. One parser per response.

In [ ]:
handler = ArticleResponseHandler()
parser = StreamingJSONParser(handler=handler)

## Feed it one character at a time

A single character is the smallest delta imaginable, so this is the hardest case the parser could
be handed. `parse_incremental` takes whatever it is given and fires whatever callbacks that
completes.

In [ ]:
for c in raw_json:
    parser.parse_incremental(c)

Formatted markdown out of a raw JSON string fed in one character at a time - and the section text
printed as it arrived, not after its field closed. This is exactly what the live stream needs.

# One function for the whole thing

Goal: stream, parse and print in a single call, with the schema and the handler as arguments.

## Wrap it up

The stream loop from above with the parser in the middle: each `delta` goes to `parse_incremental`
instead of `print`, so the handler decides what reaches the screen. `output_type` and
`parser_handler` are arguments, so one function serves any schema.

In [ ]:
def llm_structured_stream(
        user_prompt,
        output_type,
        parser_handler=JSONParserHandler(),
        instructions=None,
        model="gpt-4o-mini",
    ):
    messages = []

    if instructions:
        messages.append({
            "role": "system",
            "content": instructions
        })

    messages.append({
        "role": "user",
        "content": user_prompt
    })

    parser = StreamingJSONParser(handler=parser_handler)

    with openai_client.responses.stream(
        model="gpt-4o-mini",
        input=messages,
        text_format=output_type,
    ) as stream:
        response = None
        for event in stream:
            if hasattr(event, 'delta'):
                parser.parse_incremental(event.delta)
            if hasattr(event, 'response'):
                response = event.response

    return response

One snag worth knowing: `model` is accepted as an argument but the call below hardcodes
`"gpt-4o-mini"`, so passing a different model has no effect.

## Run it

`instructions` becomes the system message, `user_prompt` the turn. Same `ArticleResponseHandler`
as before, now fed by a live stream instead of a pasted string.

In [ ]:
instructions = "your task is to tell the user bad time stories"
user_prompt =  "unicorn"

result = llm_structured_stream(
    instructions=instructions,
    user_prompt=user_prompt,
    output_type=ArticleResponse,
    parser_handler=ArticleResponseHandler(),
)

`result` holds the finished response object, but everything worth reading already reached the
screen through the handler - nothing below this uses it.

# Streaming RAG

Goal: the pipeline from `02-rag`, with a structured answer that streams. Retrieval is unchanged -
read the repo, chunk it, keyword index - and the answer comes back as a schema instead of prose.

## Rebuild retrieval and the prompt

Everything from the RAG notebook in one cell: `gitsource` reads and chunks the Evidently docs,
`minsearch` indexes them, `search` returns five chunks and `build_prompt` stuffs them into the
template. Note `instructions` is reassigned here, so the "bad time stories" system message from
above no longer applies to anything below.

In [ ]:
import json

from gitsource import GithubRepositoryDataReader, chunk_documents
from minsearch import Index

reader = GithubRepositoryDataReader(
    repo_owner="evidentlyai",
    repo_name="docs",
    allowed_extensions={"md", "mdx"},
)
files = reader.read()

parsed_docs = [doc.parse() for doc in files]
chunked_docs = chunk_documents(parsed_docs, size=3000, step=1500)

index = Index(
    text_fields=["title", "description", "content"],
    keyword_fields=["filename"]
)
index.fit(chunked_docs)

print(f"Indexed {len(chunked_docs)} chunks from {len(files)} documents")

def search(query):
    results = index.search(
        query=query,
        num_results=5
    )
    return results



instructions = """
You're a documentation assistant. Answer the QUESTION based on the CONTEXT from our documentation.

Use only facts from the CONTEXT when answering.
If the answer isn't in the CONTEXT, say so.
"""


prompt_template = """
<QUESTION>
{question}
</QUESTION>

<CONTEXT>
{context}
</CONTEXT>
""".strip()

def build_prompt(question, search_results):
    context = json.dumps(search_results, indent=2)
    return prompt_template.format(
        question=question,
        context=context
    )

The reader downloads the repo on every run, so this is the slow cell - and the chunk count drifts
as the docs change upstream.

## Describe the answer you want

More than prose this time: the schema asks for the answer plus metadata about it - whether
anything was found, a confidence score with its reasoning, a category from a fixed list, and
follow-up questions. `Literal` is what pins `answer_type` to those five values.

In [ ]:
from typing import Literal

class RAGResponse(BaseModel):
    """
    This model provides a structured answer with metadata about the response,
    including confidence, categorization, and follow-up suggestions.
    """

    answer: str = Field(description="The main answer to the user's question in markdown")
    found_answer: bool = Field(description="True if relevant information was found in the documentation")
    confidence: float = Field(description="Confidence score from 0.0 to 1.0 indicating how certain the answer is")
    confidence_explanation: str = Field(description="Explanation about the confidence level")
    answer_type: Literal["how-to", "explanation", "troubleshooting", "comparison", "reference"] = Field(description="The category of the answer")
    followup_questions: list[str] = Field(description="Suggested follow-up questions the user might want to ask")

## Print only what matters

Three callbacks, three jobs: stream the answer text as it arrives, print `answer_type` once that
field closes, and print each follow-up question as its array item completes. The confidence fields
are left unprinted - they are still in the response object.

In [ ]:
class RAGResponseHandler(JSONParserHandler):
    def on_value_chunk(self, path: str, field_name: str, chunk: str) -> None:
        if path == '' and field_name == 'answer':
            print(chunk, end='', flush=True)

    def on_field_end(self, path: str, field_name: str, value: str, parsed_value: Any = None) -> None:
        if path == '' and field_name == 'answer_type':
            print('\nanswer type:', value)

    def on_array_item_end(self, path: str, field_name: str, item: Dict[str, Any] = None) -> None:
        if field_name == 'followup_questions':
            print('follow up question:', item)

## Chain retrieve, prompt, stream

`search`, `build_prompt` and `llm_structured_stream` in three lines. `instructions` is read from
the global set two cells up rather than passed in, so re-running the story cell above would change
what this sends.

In [ ]:
def rag(query):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    return llm_structured_stream(
        instructions=instructions,
        user_prompt=prompt,
        output_type=RAGResponse,
        parser_handler=RAGResponseHandler()
    )

## Run it end to end

One question in, a streamed answer and its metadata out.

In [ ]:
response = rag('llm as a judge')

The answer streams, then the category and follow-ups arrive at the end - they cannot come sooner,
because the model writes them after the answer. `response.output_parsed` still carries the
confidence score and `found_answer` even though nothing printed them.